# Data Types in Finance — Solutions
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> ⚠️ **This file contains complete solutions. Release to students only after the submission deadline.**


In [ ]:
!pip install yfinance seaborn --quiet
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})
YELLOW='#FDE70E'; ORANGE='#FCB310'; RED='#C70101'; GREY='#4B4B4B'

# Download shared data once
START = '2019-01-01'; END = '2024-12-31'
aapl_data    = yf.download('AAPL', start=START, end=END, auto_adjust=True, progress=False)
aapl_prices  = aapl_data['Close'].squeeze()
aapl_ret     = aapl_prices.pct_change().dropna()

tickers      = ['AAPL','MSFT','JPM','^GSPC']
raw          = yf.download(tickers, start=START, end=END, auto_adjust=True, progress=False)
prices_panel = raw['Close'].copy()
prices_panel.columns = ['AAPL','MSFT','JPM','SP500']
prices_panel = prices_panel.dropna()
ret_panel    = prices_panel.pct_change().dropna()
print('✓ Data loaded.')

---
# Solution 1 — Identify the Data Type

| # | Data Type | Justification |
|---|-----------|---------------|
| a | **Time series** | One entity (EUR/USD), many time points, order matters |
| b | **Cross-sectional** | Many entities (50 firms), one point in time (31 Dec 2023) |
| c | **Panel** | Multiple entities (3 firms) observed over multiple periods |
| d | **Time series** | One entity (Nestlé), ordered monthly observations |
| e | **Cross-sectional** | Many entities (200 banks), single snapshot (Q1 2024) |
| f | **Time series** | One entity (SMI index), ordered monthly observations |

---
# Solution 2 — Download and Explore a Swiss Stock

In [ ]:
TICKER_EX = 'NESN.SW'
data_ex   = yf.download(TICKER_EX, start=START, end=END, auto_adjust=True, progress=False)
prices_ex = data_ex['Close'].squeeze()
ret_ex    = prices_ex.pct_change().dropna() * 100

print(f'Stock: {TICKER_EX}')
print(f'Trading days: {len(prices_ex)}')
print(f'Worst day: {ret_ex.min():.2f}% on {ret_ex.idxmin().date()}')
print(f'Best day:  {ret_ex.max():.2f}% on {ret_ex.idxmax().date()}')
print(f'Daily std: {ret_ex.std():.4f}%  (AAPL: ~1.5%)')
print(f'Daily mean: {ret_ex.mean():.4f}%')

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                          gridspec_kw={'height_ratios':[2,1], 'hspace':0.05})
axes[0].plot(prices_ex.index, prices_ex.values, color='black', lw=1.3)
axes[0].set_ylabel('Price (CHF)'); axes[0].set_title(f'{TICKER_EX} — Price & Returns', fontweight='bold')
axes[1].bar(ret_ex.index, ret_ex.values,
            color=[RED if v<0 else 'black' for v in ret_ex.values], width=1.0)
axes[1].axhline(0, color=GREY, lw=0.8); axes[1].set_ylabel('Daily ret. (%)')
plt.tight_layout(); plt.show()

print('\nAnswers:')
print('2. Nestlé (~0.9% std) is less volatile than AAPL (~1.5%) — typical for a consumer staples firm')
print('3. Positive mean return = stock appreciated over the period, compensating investors for risk')

---
# Solution 3 — Simple vs. Log Returns: Manual Calculation

In [ ]:
scenarios = [('A: Small move',     100, 101,  None),
             ('B: Large positive', 100, 150,  None),
             ('C: Large negative', 100,  65,  None)]

for label, P0, P1, _ in scenarios:
    simple = (P1-P0)/P0
    log    = np.log(P1/P0)
    diff   = abs(simple - log)
    print(f'{label}: Simple={simple*100:.3f}%  Log={log*100:.3f}%  |Diff|={diff*100:.4f}%')

print('\nScenario D: Round trip $100→$80→$100')
P0_d, P1_d, P2_d = 100, 80, 100
s1=(P1_d-P0_d)/P0_d; s2=(P2_d-P1_d)/P1_d
l1=np.log(P1_d/P0_d); l2=np.log(P2_d/P1_d)
print(f'  Simple total: {((1+s1)*(1+s2)-1)*100:.4f}%  (must multiply)')
print(f'  Log total:    {(l1+l2)*100:.4f}%  (just add!) ← exactly 0')
print('\nKey: Difference largest for large moves (B and C). Log returns are ADDITIVE (D).')

---
# Solution 4 — Annualised Returns & Sharpe Ratio

In [ ]:
TRADING_DAYS = 252; RF = 0.04

ann_return = (1 + aapl_ret.mean()) ** TRADING_DAYS - 1
ann_vol    = aapl_ret.std() * np.sqrt(TRADING_DAYS)
sharpe     = (ann_return - RF) / ann_vol

print(f'AAPL (2019–2024):')
print(f'  Annualised return: {ann_return*100:.2f}%')
print(f'  Annualised vol:    {ann_vol*100:.2f}%')
print(f'  Sharpe ratio:      {sharpe:.2f}')
print('\nAnswers:')
print('1. Sharpe > 1 is considered good. Berkshire Hathaway benchmark ≈ 0.76 long-run.')
print('2. Higher RF → lower Sharpe. If RF=5%: Sharpe = (return - 0.05) / vol → decreases.')

---
# Solution 5 — COVID Crash Analysis

In [ ]:
prices_covid = aapl_prices.loc['2020-01-01':'2020-06-30']
ret_covid    = aapl_ret.loc['2020-01-01':'2020-06-30'] * 100

worst_day  = ret_covid.idxmin()
worst_ret  = ret_covid.min()
peak       = prices_covid.max(); trough = prices_covid.min()
drawdown   = (trough - peak) / peak * 100
peak_date  = prices_covid.idxmax(); trough_date = prices_covid.idxmin()
days_down  = (trough_date - peak_date).days
n_std      = worst_ret / (aapl_ret.std()*100)

print(f'Worst single day: {worst_ret:.2f}% on {worst_day.date()}')
print(f'Peak-to-trough:   {drawdown:.1f}% over {days_down} calendar days')
print(f'Worst day in std: {n_std:.1f} standard deviations below mean')
print(f'\n→ A move of {n_std:.0f}σ would be essentially impossible under normal distribution')
print('→ This illustrates FAT TAILS — extreme events are far more common than models predict')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(prices_covid.index, prices_covid.values, color='black', lw=1.5)
ax.axvline(peak_date,   color=GREY, ls=':', lw=1.2, label=f'Peak  ({peak_date.date()})')
ax.axvline(trough_date, color=RED,  ls=':', lw=1.2, label=f'Trough ({trough_date.date()})')
ax.set_title('AAPL — COVID Crash (Jan–Jun 2020)', fontweight='bold')
ax.set_ylabel('Price (USD)'); ax.legend()
plt.tight_layout(); plt.show()

---
# Solution 6 — Beta Estimation

In [ ]:
results = {}
for col in ['AAPL','MSFT','JPM']:
    slope, intercept, r, p, se = stats.linregress(ret_panel['SP500'], ret_panel[col])
    results[col] = {'Beta': round(slope,3), 'Alpha (daily %)': round(intercept*100,4), 'R²': round(r**2,3)}

beta_df = pd.DataFrame(results).T
print(beta_df)

mkt_drop = -0.02
print('\nIf S&P 500 drops 2%:')
for col in ['AAPL','MSFT','JPM']:
    exp = beta_df.loc[col,'Beta'] * mkt_drop * 100
    print(f'  {col}: {exp:.2f}%')

print('\nAnswers:')
print('2. Most defensive = lowest beta. JPM is not always lowest — depends on period.')
print('3. R²=0.63 → 63% of daily AAPL variation explained by market. Rest = company-specific news.')

---
# Solution 7 — Beta for Your Own Stock

In [ ]:
MY_STOCK = 'TSLA'
my_data  = yf.download(MY_STOCK, start=START, end=END, auto_adjust=True, progress=False)
my_ret   = my_data['Close'].squeeze().pct_change().dropna()

common   = my_ret.index.intersection(ret_panel['SP500'].index)
my_r     = my_ret.loc[common]
sp_r     = ret_panel['SP500'].loc[common]

slope, intercept, r, p, se = stats.linregress(sp_r, my_r)
print(f'{MY_STOCK}: Beta={slope:.3f}  Alpha={intercept*100:.4f}%/day  R²={r**2:.3f}')

x_line = np.linspace(sp_r.min(), sp_r.max(), 100)
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(sp_r*100, my_r*100, alpha=0.18, s=8, color='black')
ax.plot(x_line*100, (intercept + slope*x_line)*100, color=RED, lw=2,
        label=f'β={slope:.2f}, R²={r**2:.2f}')
ax.axhline(0, color=GREY, lw=0.7); ax.axvline(0, color=GREY, lw=0.7)
ax.set_xlabel('S&P 500 daily return (%)'); ax.set_ylabel(f'{MY_STOCK} daily return (%)')
ax.set_title(f'Beta regression: {MY_STOCK} vs S&P 500', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

print(f'\n→ β={slope:.2f} → {MY_STOCK} is {"aggressive" if slope>1 else "defensive"} (moves {slope:.1f}× the market)')
print('→ Scatter around line = idiosyncratic (company-specific) risk')
print('→ This analysis uses TIME SERIES data (daily returns over 5 years)')

---
# Solution 8 — Correlation Matrix

In [ ]:
corr = ret_panel.corr()
print('Correlation Matrix:'); print(corr.round(3))

fig, ax = plt.subplots(figsize=(6,5))
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, annot_kws={'size':13,'fontweight':'bold'})
ax.set_title('Correlation Matrix — Daily Returns (2019–2024)', fontweight='bold')
plt.tight_layout(); plt.show()

print('\nAnswers:')
print('1. AAPL & MSFT typically most correlated — both large-cap US tech stocks, similar drivers')
print('2. SP500 has high correlation with all — it IS the market. Individual stocks add idiosync. risk.')
print('3. Diagonal always 1.0 because a variable is perfectly correlated with itself')

---
# Solution 9 — Crisis vs. Normal Correlations

In [ ]:
crisis = ret_panel.loc['2020-02-01':'2020-05-31']
normal = ret_panel.loc['2021-01-01':'2021-12-31']

corr_crisis = crisis.corr()
corr_normal = normal.corr()

fig, axes = plt.subplots(1,2, figsize=(13,5))
for ax, corr, title in [
    (axes[0], corr_crisis, 'COVID Crisis (Feb–May 2020)'),
    (axes[1], corr_normal, 'Normal Period (2021)')
]:
    sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, vmin=-1, vmax=1, annot_kws={'size':13})
    ax.set_title(title, fontweight='bold')
plt.tight_layout(); plt.show()

# Off-diagonal average
def avg_offdiag(c):
    m = c.values; n = m.shape[0]
    return (m.sum() - np.trace(m)) / (n*(n-1))

print(f'Average off-diagonal correlation:')
print(f'  COVID crisis: {avg_offdiag(corr_crisis):.3f}')
print(f'  Normal 2021:  {avg_offdiag(corr_normal):.3f}')
print('\nAnswers:')
print('1. Correlations significantly HIGHER during COVID — all stocks fell together')
print('2. Diversification FAILS during crises — you need it most when it works least')
print('3. Called "correlation breakdown" or "flight to quality" phenomenon')

---
# Solution 10 — Equally-Weighted Portfolio

In [ ]:
stocks  = ['AAPL','MSFT','JPM']
weights = np.array([1/3,1/3,1/3])
port_ret = (ret_panel[stocks] * weights).sum(axis=1)

cum_port = (1+port_ret).cumprod()-1
cum_sp   = (1+ret_panel['SP500']).cumprod()-1

fig, ax = plt.subplots(figsize=(13,5))
ax.plot(cum_port.index, cum_port*100, color='black', lw=2, label='EW Portfolio (AAPL+MSFT+JPM)')
ax.plot(cum_sp.index,   cum_sp*100,   color=RED, lw=2, ls='--', label='S&P 500')
ax.axhline(0, color=GREY, lw=0.8)
ax.set_title('Equally-Weighted Portfolio vs. S&P 500', fontweight='bold')
ax.set_ylabel('Cumulative return (%)'); ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

RF = 0.04; TD = 252
for label, r in [('EW Portfolio', port_ret), ('S&P 500', ret_panel['SP500'])]:
    ar = (1+r.mean())**TD-1; av = r.std()*np.sqrt(TD)
    print(f'{label}: Return={ar*100:.1f}%  Vol={av*100:.1f}%  Sharpe={(ar-RF)/av:.2f}')

print('\nAnswers:')
print('1. EW portfolio of large-cap tech/finance likely outperforms broad S&P 500 in bull markets')
print('2. Higher return comes with higher vol — must consider Sharpe ratio, max drawdown, etc.')
print('3. Changing weights to 50/25/25 increases AAPL exposure → higher beta, potentially higher return AND risk')

---
# Solution — Challenge A: Rolling Beta

In [ ]:
WINDOW = 126
rolling_betas = []
rolling_dates = []

for i in range(WINDOW, len(ret_panel)):
    w = ret_panel.iloc[i-WINDOW:i]
    b, *_ = stats.linregress(w['SP500'], w['AAPL'])
    rolling_betas.append(b)
    rolling_dates.append(ret_panel.index[i])

rb = pd.Series(rolling_betas, index=rolling_dates)

fig, ax = plt.subplots(figsize=(13,4))
ax.plot(rb.index, rb.values, color='black', lw=1.5)
ax.axhline(1.0, color=RED, lw=1.2, ls='--', label='β = 1')
ax.axhline(rb.mean(), color=ORANGE, lw=1.2, ls=':', label=f'Mean β = {rb.mean():.2f}')
ax.fill_between(rb.index, rb.values, 1.0, where=rb.values>1, alpha=0.15, color=RED)
ax.set_title('AAPL Rolling 6-Month Beta vs. S&P 500', fontweight='bold')
ax.set_ylabel('Beta (β)'); ax.legend()
plt.tight_layout(); plt.show()

print(f'Beta range: {rb.min():.2f} – {rb.max():.2f}')
print(f'Max beta: {rb.idxmax().date()} (β={rb.max():.2f})')
print('\n→ Beta is NOT constant — changes with market regime')
print('→ Using a single static beta for risk management can be misleading')

---
# Solution — Challenge B: Risk-Return Plot

In [ ]:
universe = ['AAPL','MSFT','JPM','TSLA','NVDA','JNJ','XOM','AMZN','META','GS','^GSPC']
raw_u    = yf.download(universe, start=START, end=END, auto_adjust=True, progress=False)
ret_u    = raw_u['Close'].pct_change().dropna()
ret_u.columns = [t.replace('^GSPC','SP500') for t in ret_u.columns]

TD=252; RF=0.04
rr = pd.DataFrame({
    'Ann. Return (%)': [(1+ret_u[c].mean())**TD-1 for c in ret_u.columns],
    'Ann. Vol (%)':    [ret_u[c].std()*np.sqrt(TD) for c in ret_u.columns],
}, index=ret_u.columns) * 100

fig, ax = plt.subplots(figsize=(10,6))
ax.scatter(rr['Ann. Vol (%)'], rr['Ann. Return (%)'], color='black', s=80, zorder=3)
for ticker, row in rr.iterrows():
    ax.annotate(ticker, (row['Ann. Vol (%)'], row['Ann. Return (%)']),
                textcoords='offset points', xytext=(6,4), fontsize=9)
ax.axhline(0, color=GREY, lw=0.7)
ax.set_xlabel('Annual Volatility (%)'); ax.set_ylabel('Annual Return (%)')
ax.set_title('Risk-Return Trade-off (2019–2024)', fontweight='bold')
plt.tight_layout(); plt.show()
print(rr.round(1).sort_values('Ann. Return (%)', ascending=False))

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*

*⚠️ Release to students only after the submission deadline.*